# 04 - LLaMEA Noisy BBOB Thesis Experiment Dashboard

This notebook is used to monitor background runs, inspect results, and generate thesis-ready statistics and plots. 

It automatically loads checkpoints from `results/` (generated by background SLURM/cluster scripts) as well as summaries from `experiments/` (generated by local/notebook experiments).

In [ ]:
# Setup paths and imports
import sys
from pathlib import Path
# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import os
import glob
import json
import pandas as pd
import plotly.express as px
from analysis.results import load_summaries, print_experiment_summary

RESULTS_DIR = root_dir / 'results'
EXPERIMENTS_DIR = root_dir / 'experiments'
print("Dashboard environment initialized.")


## 1. Load Experiment Checkpoints & Summaries
Load incremental JSON checkpoints (`bbob_*_rep_*.json`) from `results/` or summary artifacts from `experiments/`.

In [ ]:
def load_results(results_dir, experiments_dir):
    records = []
    
    # 1. Load checkpoints from results/ folder if available
    pattern = os.path.join(results_dir, "bbob_*_rep_*.json")
    checkpoint_files = glob.glob(pattern)
    for filepath in checkpoint_files:
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
                data_no_code = {k: v for k, v in data.items() if k != "code"}
                records.append(data_no_code)
        except Exception as e:
            print(f"Warning: Failed to load {filepath}: {e}")

    # 2. Fallback / supplementary loading from experiments/ directory summaries
    if not records and experiments_dir.exists():
        summaries = load_summaries(experiments_dir)
        for s in summaries:
            best_err = s.get("best_final_error", float('inf'))
            records.append({
                "problem_id": s.get("problem_id"),
                "dim": s.get("dim"),
                "noise_std": s.get("noise_std"),
                "repetition": 1,
                "fitness": -best_err if isinstance(best_err, (int, float)) else float('-inf'),
                "best_final_error": best_err,
                "best_algorithm": s.get("best_algorithm")
            })

    if not records:
        return pd.DataFrame()
    return pd.DataFrame(records)

df = load_results(RESULTS_DIR, EXPERIMENTS_DIR)

if df.empty:
    print("No experiment checkpoints or summaries found in 'results/' or 'experiments/'.")
else:
    print(f"Successfully loaded {len(df)} experiment record(s).")
    display(df.head())


## 2. Progress Tracker & Summary Report
Print the experiment summary table for logged runs and check completion status across problems.

In [ ]:
print_experiment_summary(EXPERIMENTS_DIR)

if not df.empty and 'repetition' in df.columns:
    dim = df.iloc[0].get('dim', 'N/A')
    noise_std = df.iloc[0].get('noise_std', 'N/A')
    
    completed_counts = df.groupby('problem_id')['repetition'].count().reset_index(name='completed')
    completed_counts['status'] = completed_counts['completed'].apply(
        lambda x: "DONE" if x >= 5 else f"{x}/5 Completed"
    )
    
    print(f"\n--- Progress Tracker (Dimension: {dim}, Noise Std: {noise_std}) ---")
    display(completed_counts)


## 3. Statistical Analysis Table (Thesis Ready)
Calculate the mean, std dev, min, and max performance metrics across evaluated problems.

In [ ]:
if not df.empty:
    valid_runs = df[df['fitness'] != float('-inf')].copy()
    
    if not valid_runs.empty:
        stats = valid_runs.groupby('problem_id')['fitness'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
        stats.columns = ['BBOB Problem', 'Mean Fitness', 'Std Dev', 'Best Run', 'Worst Run', 'Repetitions']
        
        print("--- Thesis Performance Statistics ---")
        display(stats.round(4))
    else:
        print("No valid runs available to compute statistics.")
else:
    print("No data available.")


## 4. Performance Distribution Boxplots
Visualize the distribution of final algorithm fitness across evaluated BBOB problems.

In [ ]:
if not df.empty:
    valid_runs = df[df['fitness'] != float('-inf')].copy()
    if not valid_runs.empty:
        valid_runs['problem_id'] = valid_runs['problem_id'].astype(str)
        
        fig = px.box(
            valid_runs,
            x='problem_id',
            y='fitness',
            labels={'problem_id': 'BBOB Problem ID', 'fitness': 'Fitness Score (-error)'},
            title='LLaMEA Generated Algorithm Performance Distribution',
            points='all',
            color='problem_id'
        )
        
        fig.update_layout(
            template='plotly_white',
            showlegend=False
        )
        fig.show()
    else:
        print("No valid runs to plot.")
else:
    print("No data to plot.")


## 5. Inspect Generated Algorithm Code
Retrieve and inspect generated algorithm code for a specific problem checkpoint or logged summary.

In [ ]:
def inspect_code(problem_id, repetition=0, results_dir=RESULTS_DIR):
    filepath = Path(results_dir) / f"bbob_{problem_id}_rep_{repetition}.json"
    if not filepath.exists():
        print(f"No checkpoint found at: {filepath}")
        return
        
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
        print(f"=== Code for BBOB Problem {problem_id}, Repetition {repetition} (Fitness: {data.get('fitness', 'N/A')}) ===\n")
        print(data.get('code', 'No code string found in checkpoint.'))

# Example: inspect code for Problem 1, Repetition 0
inspect_code(problem_id=1, repetition=0)
